In [3]:
import utils
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import sys, os
# sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))
# from models.Carleton2022.model.mortality_functions import ReadOUTFiles

IMAGE = [
    "World",
    "CAN", "USA", "MEX", "RCAM", "BRA", "RSAM", # America
    "NAF", "WAF", "EAF", "SAF", # Africa
    "WEU", "CEU", "TUR", "UKR", # Europe
    "SEAS", "RUS", "ME", "INDIA", "KOR", "CHN", "SEAS", "INDO", "JAP", # Asia
    "OCE", "RSAS", "RSAF" # Oceania + other
]

### Regression with climate indices

#### Using Present Day ERA5

In [ ]:
era5dir = "X:/user/liprandicn/mt-comparison/data/ERA5_indices/"

IOD = xr.open_dataset(era5dir+"IOD_ERA5_historical_r1i1p1f1_gn_1940-202512_ye.nc")
N34 = xr.open_dataset(era5dir+"N34_ERA5_historical_r1i1p1f1_gn_1940-202512_ye.nc")
NAO = xr.open_dataset(era5dir+"winter_NAO_ERA5_historical_r1i1p1f1_gn_1940-202512.nc")

wdir = "X:/user/liprandicn/mt-comparison/"
regions = ["World", "RSAM", "WEU", "SEAS", "INDIA", "INDO", "OCE"]

def DetrendedMortality(region):
    mortality = utils.LoadMortality(wdir, "carleton2022/output/DEFAULT/mortality_default_SSP2_ERA5_2000-2025", range(2000,2026), region, "heat", "relative", "all", None)
    years = mortality.columns.values
    mor = mortality.values.flatten()
    coeffs = np.polyfit(years, mor, 2)
    polynomial = sum(coeffs[i] * years**(len(coeffs)-i-1) for i in range(len(coeffs)))
    return mor - polynomial

niño = N34.mean("lat").mean("lon").sel(valid_time=slice("2000", "2025")).sst.values - 273.15
dipole = IOD.mean("lat").mean("lon").sel(valid_time=slice("2000", "2025")).sst.values
oscillation = NAO.mean("lat").mean("lon").sel(valid_time=slice("2000", "2025")).msl.values

indices = {"Niño": niño, "NAO": oscillation, "IOD": dipole}

fig, ax = plt.subplots(7,3, figsize=(12,20))

for i,region in enumerate(regions):    
    for j, index in enumerate(indices.keys()):
    
        ax[i,j].scatter(indices[index], DetrendedMortality(region), color="navy")    
        ax[i,j].plot(indices[index], np.poly1d(np.polyfit(indices[index], DetrendedMortality(region), 1))(indices[index]), color="navy", alpha=0.5)    
        ax[i,j].text(0.05, 0.95, f"r={np.corrcoef(indices[index], DetrendedMortality(region))[0,1]:.2f}", transform=ax[i,j].transAxes, fontsize=10, 
                     verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        ax[0,j].set_title(index, fontsize=12)
        
    ax[i,0].set_ylabel(region)
    
plt.show()

In [ ]:
era5dir = "X:/user/liprandicn/mt-comparison/data/ERA5_indices/"
N34 = xr.open_dataset(era5dir+"N34_ERA5_historical_r1i1p1f1_gn_1940-202512_ye.nc")
wdir = "X:/user/liprandicn/mt-comparison/"
niño = N34.mean("lat").mean("lon").sel(valid_time=slice("2000", "2025")).sst.values - 273.15


def DetrendedMortality(region):
    mortality = utils.LoadMortality(wdir, "carleton2022/output/DEFAULT/mortality_default_SSP2_ERA5_2000-2025", range(2000,2026), region, "heat", "relative", "all", None)
    years = mortality.columns.values
    mor = mortality.values.flatten()
    coeffs = np.polyfit(years, mor, 2)
    polynomial = sum(coeffs[i] * years**(len(coeffs)-i-1) for i in range(len(coeffs)))
    return mor - polynomial


fig, ax = plt.subplots(5,6, figsize=(20,15))
axs = ax.flatten()

for i,region in enumerate(IMAGE):    
    
    axs[i].scatter(niño, DetrendedMortality(region))    
    axs[i].plot(niño, np.poly1d(np.polyfit(niño, DetrendedMortality(region), 1))(niño), alpha=0.5)    
    axs[i].text(0.05, 0.95, f"r={np.corrcoef(niño, DetrendedMortality(region))[0,1]:.2f}", transform=axs[i].transAxes, fontsize=10, 
                    verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    axs[i].set_title(region, fontsize=12)
        
    # ax[i,0].set_ylabel(region)
    
plt.show()

#### Using historical data 1970-2025

In [ ]:
era5dir = "X:/user/liprandicn/mt-comparison/data/ERA5_indices/"

IOD = xr.open_dataset(era5dir+"IOD_ERA5_historical_r1i1p1f1_gn_1940-202512_ye.nc")
N34 = xr.open_dataset(era5dir+"N34_ERA5_historical_r1i1p1f1_gn_1940-202512_ye.nc")
NAO = xr.open_dataset(era5dir+"winter_NAO_ERA5_historical_r1i1p1f1_gn_1940-202512.nc")

wdir = "X:/user/liprandicn/mt-comparison/"
regions = ["World", "RSAM", "WEU", "SEAS", "INDIA", "INDO", "OCE"]

def DetrendedMortality(region):
    mortality = utils.LoadMortality(wdir, "carleton2022/output/modes/IMAGE/MOR_modes_SSP2_ERA5_NoAdap_1970-2025", range(1970,2026), region, "heat", "relative", "all", None)
    years = mortality.columns.values
    mor = mortality.values.flatten()
    coeffs = np.polyfit(years, mor, 2)
    polynomial = sum(coeffs[i] * years**(len(coeffs)-i-1) for i in range(len(coeffs)))
    return mor - polynomial

niño = N34.mean("lat").mean("lon").sel(valid_time=slice("1970", "2025")).sst.values - 273.15
dipole = IOD.mean("lat").mean("lon").sel(valid_time=slice("1970", "2025")).sst.values
oscillation = NAO.mean("lat").mean("lon").sel(valid_time=slice("1970", "2025")).msl.values

indices = {"Niño": niño, "NAO": oscillation, "IOD": dipole}

fig, ax = plt.subplots(7,3, figsize=(12,20))

for i,region in enumerate(regions):    
    for j, index in enumerate(indices.keys()):
    
        ax[i,j].scatter(indices[index], DetrendedMortality(region), color="navy")    
        ax[i,j].plot(indices[index], np.poly1d(np.polyfit(indices[index], DetrendedMortality(region), 1))(indices[index]), color="navy", alpha=0.5)    
        ax[i,j].text(0.05, 0.95, f"r={np.corrcoef(indices[index], DetrendedMortality(region))[0,1]:.2f}", transform=ax[i,j].transAxes, fontsize=10, 
                     verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        ax[0,j].set_title(index, fontsize=12)
        
    ax[i,0].set_ylabel(region)
    
plt.show()

In [ ]:
era5dir = "X:/user/liprandicn/mt-comparison/data/ERA5_indices/"
N34 = xr.open_dataset(era5dir+"N34_ERA5_historical_r1i1p1f1_gn_1940-202512_ye.nc")
wdir = "X:/user/liprandicn/mt-comparison/"
niño = N34.mean("lat").mean("lon").sel(valid_time=slice("1970", "2025")).sst.values - 273.15


def DetrendedMortality(region):
    mortality = utils.LoadMortality(wdir, "carleton2022/output/modes/IMAGE/MOR_modes_SSP2_ERA5_NoAdap_1970-2025", range(1970,2026), region, "heat", "relative", "all", None)
    years = mortality.columns.values
    mor = mortality.values.flatten()
    coeffs = np.polyfit(years, mor, 2)
    polynomial = sum(coeffs[i] * years**(len(coeffs)-i-1) for i in range(len(coeffs)))
    return mor - polynomial


fig, ax = plt.subplots(5,6, figsize=(20,15))
axs = ax.flatten()

for i,region in enumerate(IMAGE):    
    
    axs[i].scatter(niño, DetrendedMortality(region))    
    axs[i].plot(niño, np.poly1d(np.polyfit(niño, DetrendedMortality(region), 1))(niño), alpha=0.5)    
    axs[i].text(0.05, 0.95, f"r={np.corrcoef(niño, DetrendedMortality(region))[0,1]:.2f}", transform=axs[i].transAxes, fontsize=10, 
                    verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    axs[i].set_title(region, fontsize=12)
        
    # ax[i,0].set_ylabel(region)
    
plt.show()